# Few-shot Generation

## Overview

This notebook handles the first stage of the few-shot pipeline. Instead of giving the model an empty context, each prompt is prepended with one or more curated LilyPond examples. These examples define the style, structure, and constraints we want the model to follow. The job of this notebook is to assemble those few-shot prompts, run the model, and save every raw output.

All base prompts come from a single text file, and the few-shot examples come from a separate template file. This keeps the logic simple: you can swap or expand the examples without touching the code, and you can modify the prompt set independently of the examples.

Just like in the zero-shot pipeline, evaluation happens in its own notebook: 02_few_shot_evaluate.ipynb. That notebook checks compilation validity, musical structure, constraint adherence, and all metrics. Keeping generation separate means the outputs stay fixed and reproducible, even if you later refine the evaluation scripts.

## Objectives

- Provide a reproducible few-shot generation workflow.

- Combine predefined examples with each prompt in a consistent template.

- Run the model on the resulting few-shot prompts and store every generated sample.

- Produce structured output files to be consumed later by 02_few_shot_evaluate.ipynb.


## Notebook Structure

This notebook is organized into steps that:

1. Load the base prompts and the few-shot example template.

2. Assemble the final few-shot prompts to be sent to the model.

3. Initialize the OpenAI client for HuggingFace router models.

4. Generate LilyPond code for each few-shot prompt.

5. Save all raw generations in a consistent directory hierarchy.

6. Record metadata for reproducibility and later evaluation.

## Requirements

- **Python 3.8+**
- **HF_TOKEN**: HuggingFace API token (set as environment variable)
- **Model Access**: Access to the selected model via HuggingFace router

## Key Parameters

- **Model** – which model to query (e.g., GPT-OSS-20B)

- **Temperature** – randomness control (e.g., 0.7 for moderate variability)

- **Max Tokens** – upper limit of the generated sequence

- **Number of Runs** – how many samples to generate for each prompt


## Output

All raw generations are saved under:

`few_shot_outputs/raw`

These files are later loaded and evaluated in 02_few_shot_evaluate.ipynb, which handles compilation checks, musical metrics, structural analysis, and summary tables.

In [1]:
from openai import OpenAI
from IPython.display import display, Markdown
from pathlib import Path
from datetime import datetime
import os, re, json


# Helper: extract LilyPond code only
def extract_lilypond_code(text: str) -> str:
    """Extract LilyPond code from a model reply, removing markdown fences/preambles."""
    text = re.sub(r'```(?:lilypond)?\s*\n?(.*?)```', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'^(?:Here|Here\'s|Here is|Output|LilyPond code:|The following).*?:\s*',
                  '', text, flags=re.IGNORECASE)
    return text.strip()


# Generator (saves in few_shot_outputs/raw/<subfolder>)
def run_generate(
    prompt_file: str | Path = "../configs/prompts/few_shot.txt",
    *,
    model: str = "openai/gpt-oss-20b:fireworks-ai",
    num_runs: int = 2,
    max_tokens: int = 15000,
    temperature: float = 0.7,
    output_folder_name: str = "session_01"
):

    prompt_path = Path(prompt_file)
    if not prompt_path.exists():
        display(Markdown(f"**❌ Error:** Prompt file not found: `{prompt_path}`"))
        return []

    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        display(Markdown("**❌ Error:** `HF_TOKEN` environment variable not set!"))
        return []

    client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)

    # Detect base directory safely (handles notebooks too)
    try:
        base_dir = Path(__file__).resolve().parent
    except NameError:
        base_dir = Path.cwd()  # fallback for Jupyter / REPL

    save_dir = base_dir / "few_shot_outputs" / "raw" / output_folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    prompt_text = prompt_path.read_text(encoding="utf-8").strip()
    display(Markdown(f"### 📝 Input Prompt\n```\n{prompt_text}\n```\n---"))

    all_results = []
    for i in range(1, num_runs + 1):
        display(Markdown(f"## 🔄 Run {i}/{num_runs}"))
        t0 = datetime.now()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt_text}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        dt = (datetime.now() - t0).total_seconds()

        choice = resp.choices[0]
        raw_text = (choice.message.content or "") if choice.message else ""
        lp = extract_lilypond_code(raw_text)

        # Display output (no checks)
        md = f"### Model Output\n```lilypond\n{lp}\n```\n"
        md += f"⏱️ Time: {dt:.2f}s | Tokens: {resp.usage.total_tokens if resp.usage else '-'}"
        display(Markdown(md))
        display(Markdown("---"))

        # Save both raw and extracted
        (save_dir / f"raw_{i:02d}.txt").write_text(raw_text, encoding="utf-8")
        (save_dir / f"out_{i:02d}.ly").write_text(lp, encoding="utf-8")

        # Append metadata for summary.json
        all_results.append({
            "run": i,
            "elapsed_seconds": round(dt, 3),
            "tokens_used": resp.usage.total_tokens if resp.usage else None,
            "model": model,
            "temperature": temperature,
            "prompt_file": str(prompt_path)
        })

    # Write summary.json
    summary_path = save_dir / "summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    display(Markdown(f"✅ Saved all outputs and summary in `{save_dir}`"))
    return all_results

# Example:
# results = run_generate(
#     "../configs/prompts/few_shot.txt",
#     num_runs=3,
#     output_folder_name="test_batch_A"
# )

------------

Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

# Single Voice

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Key</h3>
</div>

##  C Major

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key do \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="c_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key do \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    \new Staff {
      \key do \major
      \tempo 4 = 120
      r4 do'8 re'8 mi'8 fa'8 |
      sol'4 la'8 sol'8 fa'8 mi'8 |
      re'4 do'8 re'8 mi'8 fa'8 |
      sol'4 la'8 sol'8 fa'8 mi'8 |
      re'4 do'8 re'8 mi'8 dod'8 |
      sol'4 la'8 sol'8 fa'8 mi'8 |
      do'8 re'16 mi'16 fa'16 sol'16 la'16 si'16 do''16 re''16 |
      sol''4 re''8 do''8 r4 |
    }
  >>
}
```
⏱️ Time: 16.13s | Tokens: 3308

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key do \major
    \time 4/4
    \clef treble

    \tempo "Andante"

    do'8 re'8 mi'8 fa'8 | sol'8 la'8 do''8 re''8 |
    re''8 mi''8 fa''8 sol''8 | la''8 do'''8 re'''8 mi'''8 |

    do'8 mi'8 sol'8 mi'8 | re'8 do'8 la'8 sol'8 |
    do''8 mi''8 sol''8 mi''8 | re''8 do''8 la''8 sol''8 |

    \bar "|."
  }
}
```
⏱️ Time: 9.01s | Tokens: 2061

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\c_major`

-----------------

##  G Major (1♯)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sol \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="g_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sol \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \key sol \major
    \time 4/4
    \clef treble

    sol'4 la'4 si'4 re''4
    re''8 mi''16 fa''16 sol''8 sol''8 la''8. sol''16
    sol''4 r4 r4 r4
    re''8 re''8 re''8 re''8 sol''8 la''8 re''8 sol''8

    la''4 si''4 la''4 sol''4
    la''8 si''8 la''8 sol''8 fa''8 sol''8 la''8 sol''8
    re''4 r4 r4 r4
    sol''8 sol''8 sol''8 sol''8 si''8 sol''8 fa''8 sol''8

    re''8 re''8 re''8 re''8 re''8 re''8 re''8 re''8
    sol''8 dod''8 sol''8 dod''8 sol''8 dod''8 sol''8 dod''8
    re''4 r4 r4 r4
    sol''2 r2
  }
}
```
⏱️ Time: 19.18s | Tokens: 3248

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    \new Staff \with { \clef treble } {
      \key sol \major
      R2.*4
      sol''4 la''8 sol''8 si''8 sol''8 la''8 sol''8
      r2
      re''8 re''8 re''8 re''8 si''8 re''8 si''8 re''8
      r2
      la''16 sol''16 fa''16 la''16 sol''16 la''16 sol''16 fa''16
      r2
      re''8 la''8 re''8 la''8 re''8 la''8 re''8 la''8
      r2
    }
  >>
}
```
⏱️ Time: 9.38s | Tokens: 2029

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\g_major`

----------------

##  A Minor (0)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key la \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="a_minor" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key la \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key la \minor
    \time 4/4

    R1*2

    la'8 la'8 la'8 la'8 si'8 la'8 sol'8 sol'8
    fa'8 fa'8 fa'8 fa'8 la'8 fa'8

    R2.*4

    re'8 re'8 re'8 re'8 fa'8 re'8
    dod'8 dod'8 dod'8 dod'8 fa'8 dod'8
    dod'!8 dod'8 dod'8 dod'8 fa'8 dod'8

    sol'4 sol'4 sol'2

    R2.*3

    la'8 la'8 la'8 la'8 si'8 la'8 sol'8 sol'8
    fa'8 fa'8 fa'8 fa'8 la'8 fa'8

    R2.

    re'8 re'8 re'8 re'8 fa'8 re'8
    dod'8 dod'8 dod'8 dod'8 fa'8 dod'8
  }
}
```
⏱️ Time: 6.74s | Tokens: 1679

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \key la \minor
    \time 4/4

    % Intro
    R2.*2
    la''8 la''8 la''8 la''8
    re''8 mi''8 fa''8 sol''8
    la''4 si''8 la''8 sol''8 fa''8
    mi''8 re''8 la''8 la''8

    R2.*2

    % Verse
    la''4 la''4 la''4 la''4
    re''8 mi''8 fa''8 sol''8 la''8 si''8
    re''4 re''4 re''4 re''4
    la''8 la''8 la''8 la''8 re''8 mi''8

    R4 r4 r4 r4

    % Chorus
    la''4 la''4 la''4 la''4
    si''8 la''8 sol''8 fa''8 mi''8 re''8
    la''4 la''4 la''4 la''4
    re''8 mi''8 fa''8 sol''8 la''8 si''8

    \bar "|."
  }
}
```
⏱️ Time: 11.06s | Tokens: 2127

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\a_minor`

----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Time Signature</h3>
</div>

##  3/4 (simple triple)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 3/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="three_four" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 3/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \time 3/4
    r2. \break
    do'4 re' mi' \break
    sol'4 la' si' \break
    re''4 mi'' fa'' \break
    sol''4 la'' si'' \break
    re''4 re'' re'' \break
    la''4 la'' la'' \break
    do''4 re'' mi'' \break
    sol''4 la'' si'' \break
    re''4 mi'' fa'' \break
    sol''4 la'' re'' \break
    fad'8 re'8 mi'4 \break
    re''8 re''8 re''4 \bar "|."
  }
}
```
⏱️ Time: 13.71s | Tokens: 2596

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    \new Staff <<
      \time 3/4
      \tempo "Allegro moderato" 4=120
      \repeat volta 2 {
        r4 do'8 re'8 mi'8 |
        fa'4 sol'8 la'8 si'8 |
        re'4 mi'8 fa'8 sol'8 |
        la'4 si'8 do''8 re''8 |
        mi''4 re''8 la'8 sol'8 |
        fa'4 mi'8 re'8 do'8 |
        si'4 la'8 sol'8 fa'8 |
        sol'4 fa'8 re'8 mi'8 |
      }
      \alternative {
        { r4 r8 r8 r8 | \bar "|." } 
        { r4 do''8 re''8 mi''8 | \bar "|." } 
      }
    >>
  >>
}
```
⏱️ Time: 18.67s | Tokens: 3377

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\three_four`

-------------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Harmony </h3>
</div>

##  Dyads (2-note chords)

prompt:
Write a LilyPond passage consisting only of dyads (2-note chords), not single notes.
Every event must be a dyad written using LilyPond chord brackets < >.
Each dyad must contain exactly two notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="dyads" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage consisting only of dyads (2-note chords), not single notes.
Every event must be a dyad written using LilyPond chord brackets < >.
Each dyad must contain exactly two notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do' re'>4 <mi' sol'>8 <fa' la'>8 <si' do''>4
    <re'' mi''>2
    <re' mi'>8 <fa' sol'>8 <la' si'>8 <do'' re''>8
    <mi'' fa''>4 <sol'' la''>4
    <do'' re''>4 <mi'' sol''>8 <fa'' la''>8 <si' do''>4
    <re' mi'>2
  }
}
```
⏱️ Time: 9.69s | Tokens: 1563

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do' re'>4 <mi' fa'>4 <sol' la'>4 <si' do''>4
    <do' mi'>8 <re' sol'>8 <fa' la'>8 <si' do''>8
    <do' re'>2 <mi' fa'>2
    <sol' la'>1
    <re' mi'>4 <do' sol'>4 <fa' re'>4 <mi' si'>4
    <re' do'>8 <mi' fa'>8 <sol' la'>8 <si' do''>8
  }
}
```
⏱️ Time: 17.58s | Tokens: 3230

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\dyads`

------------

##  Triads (3-note chords)

prompt:
Write a LilyPond passage consisting only of triads (3-note chords), not single notes.
Every event must be a triad written using LilyPond chord brackets < >.
Each triad must contain exactly three notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="triads" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage consisting only of triads (3-note chords), not single notes.
Every event must be a triad written using LilyPond chord brackets < >.
Each triad must contain exactly three notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do mi sol>4 <re fa la>4 <mi sol si>4 <fa la do>4 <sol si re>4 <la do mi>4 <si re fa>4 <do mi sol>4
  }
}
```
⏱️ Time: 6.83s | Tokens: 1827

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do re mi>4 <re mi fa>4 <mi fa sol>4 <fa sol la>4
    <sol la si>4 <la si do'>4 <si do' re'>4 <do' re' mi'>4
    <re' mi' fa'>4 <mi' fa' sol'>4 <fa' sol' la'>4 <sol' la' si'>4
    <la' si' do''>4 <si' do'' re''>4 <do'' re'' mi''>4 <re'' mi'' fa''>4
  }
}
```
⏱️ Time: 10.62s | Tokens: 2076

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\triads`

-------------------------------------------

# Two-Voice

##  C Major

prompt:
Write a LilyPond passage with exactly two voices, using the structure:
<< { ... } \\ { ... } >>.

The music must be as detailed and expressive as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key do \major and \time 4/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to the examples but do not copy them.

In [2]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="two_voice_c_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage with exactly two voices, using the structure:
<< { ... } \\ { ... } >>.

The music must be as detailed and expressive as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key do \major and \time 4/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to the examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \key do \major
    \time 4/4
    <<
      {
        do''4 \p
        re''8[ sol''16 la''16 fa''8 mi''8] si''4 r16
        re''4 \> la''4[ sol''4] mi''4
        do''8 re''16 mi''16 fa''8 sol''8 la''8 si''4 r16
        re''4~ re''8[ la''8 sol''8] re''8- \> si''4 \fermata
      }
      \\
      {
        do4 \p
        re8[ sol16 la16 fa8 mi8] si8 r16
        re4 \> la4[ sol4] mi4
        do8 re16 mi16 fa8 sol8 la8 si4 r16 r8
        re4~ re8[ la8 sol8] re8- \> si4 \fermata
      }
    >>
  }
}
```
⏱️ Time: 41.18s | Tokens: 7007

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    {
      \key do \major
      \time 4/4

      \pp
      \staccato ( do''8 re''8 mi''8 fa''8 )
      sol''8[ sol''8] \staccato
      la''8[ la''8] \staccato

      \mp
      do''8[ re''8 mi''8 fa''8] \accent
      sol''8[ sol''8] \accent
      la''8[ la''8] \accent

      \mf
      do''8[ re''8] \staccato
      sol''8[ sol''8 la''8 fa''8] \staccato
      mi''8[ re''8 mi''8 fa''8] \accent

      \f
      do''8[ re''8] \staccato
      sol''8[ sol''8 la''8 fa''8] \staccato
      mi''8[ re''8 mi''8 fa''8] \accent

      \dim
      do''8~ do''8 \staccato \!

      \cresc
      do''8[ re''8 mi''8 fa''8] \staccato \!

      R4
    }
    \\ {
      \key do \major
      \time 4/4

      \pp
      \staccato ( do8 re8 mi8 fa8 )
      sol8[ sol8] \staccato
      la8[ la8] \staccato

      \mp
      do8[ re8 mi8 fa8] \accent
      sol8[ sol8] \accent
      la8[ la8] \accent

      \mf
      do8[ re8] \staccato
      sol8[ sol8 la8 fa8] \staccato
      mi8[ re8 mi8 fa8] \accent

      \f
      do8[ re8] \staccato
      sol8[ sol8 la8 fa8] \staccato
      mi8[ re8 mi8 fa8] \accent

      \dim
      do8~ do8 \staccato \!

      \cresc
      do8[ re8 mi8 fa8] \staccato \!

      R4
    }
  >>
}
```
⏱️ Time: 47.68s | Tokens: 8074

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\few_shot_outputs\raw\two_voice_c_major`

---------------------------------

##  G Major (1♯)

prompt:
Write a LilyPond passage with exactly two voices, using the structure:
<< { ... } \\ { ... } >>.

The music must be as detailed and expressive as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sol \major and \time 4/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to the examples but do not copy them.


In [3]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="two_voice_g_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage with exactly two voices, using the structure:
<< { ... } \\ { ... } >>.

The music must be as detailed and expressive as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sol \major and \time 4/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to the examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"
\score {
  <<
    { \key sol \major \time 4/4 \mf
      sol''4\accent la''4\accent si''4 do'''4
      | re'''4\accent mi'''4 fa'''4 sol'''4
      | la'''8[ sol'''8] re'''8 mi'''8 sol'''8[ si'''8] la'''8 do'''8
      | do'''4. si''8 la''8 sol''8 sol''4
      | sol''8[ fa''8 la''8] sol''8[ si''8] mi''8[ re''8]
      | re''4 ~ re''8 mi''8 fa''8 sol''8 sol''4
      | la''4 ~ la''8 sol''8 re''8 mi''8 mi''4
      | sol''4 ~ sol''8 mi''8 la''8 sol''8 sol''4
    }
    { \pp
      sol4 la4 si4 do'4
      | re'4 mi'4 fa'4 sol'4
      | la'8[ sol'8] re'8 mi'8 la'8[ sol'8] re'8 mi'8 do'8
      | do'4 ~ do'8 si8 la8 sol8 sol4
      | sol8[ fa8 la8] sol8[ si8] mi8[ re8] do8
      | re4 ~ re8 mi8 fa8 sol8 sol4
      | la4 ~ la8 sol8 re8 mi8 mi4
      | sol4 ~ sol8 mi8 la8 sol8 sol4
    }
  >>
}
```
⏱️ Time: 39.46s | Tokens: 8044

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    { \key sol \major \time 4/4
      (sol''4. la''8) \p
      fa''16 sol''16 sol''8. la''16 \> \sustain
      sol''8 ~ sol''8 re''8 mi''8. mi''16 \f
      sol''2
      re''4. mi''8 \p
      fa''4 sol''8. la''16 \! \sustain
      la''4 la''4 \pp
      sol''2 \f
    }
    { \key sol \major \time 4/4
      sol''4 sol''4 \pp
      re''4 mi''4 \p
      sol''4 la''4 \p
      sol''2 \p
      re''4 mi''4 \pp
      sol''4 fa''4 \p
      sol''4 sol''4 \p
      sol''2 \f
    }
  >>
}
```
⏱️ Time: 17.40s | Tokens: 3461

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\few_shot_outputs\raw\two_voice_g_major`

------------------------------

##  A Minor (0)

prompt:
Write a LilyPond passage with exactly two voices, using the structure:
<< { ... } \\ { ... } >>.

The music must be as detailed and expressive as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key la \minor and \time 4/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to the examples but do not copy them.


In [2]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="two_voice_a_minor" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage with exactly two voices, using the structure:
<< { ... } \\ { ... } >>.

The music must be as detailed and expressive as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key la \minor and \time 4/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to the examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

<<
{
  \key la \minor
  \time 4/4
  \p
  ( la''8 si''8 si''8 la''8 )
  sol''8 fa''8 mi''8 re''8
  \f
  ( re''8 re''8 re''8 re''8 )
  mi''8 mi''8 mi''8 mi''8
  \p
  ( la''8 la''8 la''8 la''8 )
  sol''8 sol''8 sol''8 sol''8
  \f
  ( fa''8 fa''8 fa''8 fa''8 )
  re''8 re''8 re''8 re''8
}
\\
{
  \key la \minor
  \time 4/4
  la'4 r4 r4 r4
  re'4 r4 r4 r4
  mi'4 r4 r4 r4
  fa'4 r4 r4 r4
}
>>
```
⏱️ Time: 17.21s | Tokens: 3901

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond

```
⏱️ Time: 68.13s | Tokens: 10988

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\few_shot_outputs\raw\two_voice_a_minor`

-------------------------------

##  3/4 (simple triple)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 3/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [2]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="two_voice_three_four" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage with exactly two voices, using the structure:
<< { ... } \\ { ... } >>.

The music must be as detailed and expressive as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 3/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to the examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \time 3/4
  <<
    {
      do''4 re''8[ re''16 si''16] r4
      mi''4. fad''8[ fa''16] r4
      sol''4 sol''8[ sol''16 fa''16] r4
      la''4 la''8[ la''16 sol''16] r4
      do'''4 re'''8[ re'''16 si'''16] r4
      mi'''4. fad'''8[ fa'''16] r4
      sol'''4 sol'''8[ sol'''16 fa'''16] r4
      la'''4 la'''8[ la'''16 sol'''16] r4
    }
    \\
    {
      [do'' re'' mi'']4 r4 r4
      [do'' re'' mi'']4 r4 r4
      [do'' re'' mi'']4 r4 r4
      [do'' re'' mi'']4 r4 r4
      [do''' re''' mi''']4 r4 r4
      [do''' re''' mi''']4 r4 r4
      [do''' re''' mi''']4 r4 r4
      [do''' re''' mi''']4 r4 r4
    }
  >>
}
```
⏱️ Time: 46.26s | Tokens: 6917

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    {
      \time 3/4
      \p
      R4. r8 r8 r8
      [mi'' sol'' la'']4 r4 r4
      ( mi''4~ mi''8[ re''8 la''8] ) r8
      \pp
      ( sol''4~ sol''8[ fa''8 mi''8] ) r8
      \p
      ( la''4~ la''8[ sol''8 fa''8] ) r8
      \f
      mi''8[ re''8 do''8] r8 r8 r8
      \f
      re''8[ sol''8 la''8] r8 r8 r8
      r4 r4 r4
    }
    \\
    {
      \time 3/4
      \p
      R4. r8 r8 r8
      [re'' sol'' la'']4 r4 r4
      ( re''4~ re''8[ do''8 sol''8] ) r8
      \pp
      ( la''4~ la''8[ si''8 dod''8] ) r8
      \p
      ( sol''4~ sol''8[ fa''8 mi''8] ) r8
      \f
      re''8[ do''8 mi''8] r8 r8 r8
      \f
      do''8[ sol''8 la''8] r8 r8 r8
      r4 r4 r4
    }
  >>
}
```
⏱️ Time: 102.18s | Tokens: 11104

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\few_shot_outputs\raw\two_voice_three_four`